In [ ]:
import tifffile as tiff
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model


# Load Model
model = load_model("/content/unet_ct_denoising.keras", compile=False)


# Load Image
img = tiff.imread("/content/sample_image.tif")

# Normalize (0–1 range)
img = img.astype(np.float32)
img = (img - img.min()) / (img.max() - img.min())


# Add Noise (simulation)
noise = np.random.normal(0, 0.05, img.shape)
noisy = np.clip(img + noise, 0, 1)

# Patch-Based Reconstruction (SLIDING WINDOW)
patch_size = 64
step = 32

recon = np.zeros_like(noisy)
count = np.zeros_like(noisy)

for x in range(0, noisy.shape[0] - patch_size, step):
    for y in range(0, noisy.shape[1] - patch_size, step):

        patch = noisy[x:x+patch_size, y:y+patch_size]
        patch_input = patch[np.newaxis, ..., np.newaxis]

        pred = model.predict(patch_input, verbose=0)[0, :, :, 0]

        recon[x:x+patch_size, y:y+patch_size] += pred
        count[x:x+patch_size, y:y+patch_size] += 1

# Avoid division error
recon = recon / (count + 1e-8)

# Visualization
plt.figure(figsize=(15,5))

plt.subplot(1,3,1)
plt.imshow(img, cmap='gray')
plt.title("Original")
plt.axis('off')

plt.subplot(1,3,2)
plt.imshow(noisy, cmap='gray')
plt.title("Noisy Input")
plt.axis('off')

plt.subplot(1,3,3)
plt.imshow(recon, cmap='gray')
plt.title("Denoised Output (Reconstructed)")
plt.axis('off')

plt.show()

# Metrics (PSNR + SSIM + Baselines)

In [ ]:
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
import numpy as np

# ensured same dtype
img = img.astype(np.float32)
noisy = noisy.astype(np.float32)
gaussian = gaussian.astype(np.float32)
median = median.astype(np.float32)
model_output = model_output.astype(np.float32)

# PSNR
print("Noisy PSNR:", psnr(img, noisy))
print("Gaussian PSNR:", psnr(img, gaussian))
print("Median PSNR:", psnr(img, median))
print("Model PSNR:", psnr(img, model_output))

# SSIM
print("Noisy SSIM:", ssim(img, noisy, data_range=1.0))
print("Gaussian SSIM:", ssim(img, gaussian, data_range=1.0))
print("Median SSIM:", ssim(img, median, data_range=1.0))
print("Model SSIM:", ssim(img, model_output, data_range=1.0))

# Comparison Figure
plt.figure(figsize=(12,4))

plt.subplot(1,4,1)
plt.imshow(img, cmap='gray')
plt.title("Ground Truth")
plt.axis('off')

plt.subplot(1,4,2)
plt.imshow(noisy, cmap='gray')
plt.title("Noisy")
plt.axis('off')

plt.subplot(1,4,3)
plt.imshow(gaussian, cmap='gray')
plt.title("Gaussian")
plt.axis('off')

plt.subplot(1,4,4)
plt.imshow(model_output, cmap='gray')
plt.title("U-Net Output")
plt.axis('off')

plt.show()
